# AF2·03 — The Pair Representation & Triangle Operations

**Mechanism of the day:** the geometric-consistency engine. This is the notebook that
finally kills the indirect-coupling problem from rung 01 — the phantom `A–C` contact
that mutual information could not tell apart from a real one.

Recall the trap. `A` contacts `B`, `B` contacts `C`, so `A` and `C` end up correlated
*through* `B` — and any **pairwise** method (MI, APC, a per-pair MLP) is helpless,
because it scores `(A,C)` in isolation, never asking whether that correlation is
already *explained* by the other two edges.

The fix has to reason about **triples**. AlphaFold's pair representation `z[i,j]` is
updated by two operations that both march over a third residue `k`:

- the **triangle multiplicative update** — `z[i,j]` is recomputed from `z[i,k]` and
  `z[j,k]` summed over all `k`. Every pair is rebuilt from the triangles it belongs
  to. This is a soft, learned version of the triangle inequality: an edge must be
  consistent with the two edges that close its triangle.
- **triangle attention** — attention over pairs whose logits are *biased by the third
  edge* `z[j,k]`, so who-attends-to-whom respects the triangle geometry.

Before those, one more piece: how does the MSA write into the pair representation in
the first place? The **outer-product mean**. You will build all three, then train a
tiny pair refiner and watch it do what rung 01 could not — **suppress indirect
couplings while keeping real contacts** — generalising across many toy proteins it has
never seen.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions
at the bottom. Trains in ~15s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False; plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

Q = 20; L = 24; Nseq = 400          # smaller proteins so we can make MANY of them
print('toy proteins of length', L, ', MSA depth', Nseq)

## Part 1 — the MSA writes into the pair: outer-product mean

The pair representation starts life informed by the MSA. The **outer-product mean** is
how: for each residue pair `(i, j)`, take the outer product of their MSA feature
vectors and **average over the sequences**.

$$
\mathrm{opm}[i, j] = \operatorname*{mean}_{s}\ m[s, i] \otimes m[s, j]
$$

Why an outer product? Because it captures *joint* variation. If residues `i` and `j`
coevolve, the pattern of which `(feature-of-i, feature-of-j)` combinations co-occur is
exactly what the outer product records — it is the learned, multi-channel cousin of the
mutual-information joint frequency `f_ij` you built in rung 01.

### Rep 1 — `outer_product_mean(m)`
`m` is the MSA representation `[N, L, c]`. Return `[L, L, c, c]`: for each pair
`(i, j)`, the mean over the `N` sequences of the outer product `m[:, i]  ⊗  m[:, j]`.

In [ ]:
def outer_product_mean(m):
    '''MSA representation [N, L, c] -> pair signal [L, L, c, c] via mean outer product.'''
    # YOUR CODE HERE
    # hint: einsum('nic,njd->ijcd', m, m) / N
    raise NotImplementedError

# --- checkpoint ---
Nn, c = 300, 3
opm = outer_product_mean(torch.randn(Nn, L, c))
assert opm.shape == (L, L, c, c), 'wrong shape'
# columns 0,1 perfectly correlated; columns 2,3 independent. With one raw feature,
# opm[i,j] is E[col_i * col_j] — big for a correlated pair, ~0 for an independent one.
sig = torch.randn(2000, 1)
cols = torch.cat([sig, sig, torch.randn(2000, 1), torch.randn(2000, 1)], dim=1)   # [N, 4]
o = outer_product_mean(cols[:, :, None])                 # [4, 4, 1, 1]
assert o[0, 1].abs().item() > 0.5, 'a correlated pair should have a strong joint signal'
assert o[2, 3].abs().item() < 0.2, 'an independent pair should have ~zero joint signal'
print('outer-product mean ok — correlated pair %.2f vs independent %.2f'
      % (o[0, 1].item(), o[2, 3].item()),
      '| the multi-channel version of rung 01 joint frequency ✓')

## Part 2 — the triangle multiplicative update

Here is the core idea of the whole trunk. To update the edge `(i, j)`, walk over every
third residue `k` and combine the two edges that, together with `(i, j)`, form a
triangle: `(i, k)` and `(j, k)`.

$$
\begin{aligned} \text{outgoing:}&\quad \mathrm{out}[i,j] = \sum_k a[i,k]\,b[j,k] \\ \text{incoming:}&\quad \mathrm{out}[i,j] = \sum_k a[k,i]\,b[k,j] \end{aligned}
$$

where `a` and `b` are two gated linear projections of the current pair rep. Read the
outgoing rule: the new `(i, j)` is large only where residue `i` and residue `j` *share*
a strong connection to some common `k`. That is the triangle inequality made
differentiable — an edge is reinforced or suppressed based on whether the triangles it
sits in are consistent.

This single operation is why AF2 escapes the indirect-coupling trap: given strong
`(A,B)` and `(B,C)`, the update has the machinery to recognise that `(A,C)` is
*explained* by them and need not be a contact itself.

### Rep 2 — `triangle_update(a, b)`
`a, b` are `[B, L, L, c]` (batch of pair maps, already projected). Return
`[B, L, L, c]` equal to the **outgoing + incoming** combination above, summed over the
third vertex `k`. Two `einsum`s.

In [ ]:
def triangle_update(a, b):
    '''Outgoing (sum_k a_ik b_jk) + incoming (sum_k a_ki b_kj) over the third vertex k.'''
    # YOUR CODE HERE
    # hint: outgoing = einsum('bikc,bjkc->bijc', a, b)
    #       incoming = einsum('bkic,bkjc->bijc', a, b)
    raise NotImplementedError

# --- checkpoint ---
B_, cc = 1, 4
a = torch.zeros(B_, L, L, cc); b = torch.zeros(B_, L, L, cc)
# plant a chain: edges (0,1) and (1,2) are hot, everything else zero
a[0, 0, 1] = 1.0; a[0, 1, 0] = 1.0
b[0, 2, 1] = 1.0; b[0, 1, 2] = 1.0
out = triangle_update(a, b)
assert out.shape == (B_, L, L, cc)
# the (0,2) edge must light up — it is closed by the triangle through k=1 —
# while a pair with no supporting triangle, e.g. (0,3), stays dark
assert out[0, 0, 2].abs().sum() > 0.5, 'the triangle A-B-C must write into edge (0,2)'
assert out[0, 0, 3].abs().sum() < 1e-6, 'a pair with no supporting triangle stays zero'
print('triangle update ok — edges (0,1)+(1,2) propagate into (0,2), (0,3) untouched ✓')
print('That propagation across a shared vertex is exactly what pairwise MI could not do.')

## Part 3 — triangle attention

The multiplicative update mixes pairs; triangle **attention** does the same job with an
attention mechanism, so the mixing is content-dependent. In *starting-node* attention we
fix residue `i` and let its edges `(i, j)` attend over `(i, k)` — but the attention
logit is **biased by the third edge** `z[j, k]`:

$$
\mathrm{logit}[i,j,k] = \frac{q[i,j]\cdot k[i,k]}{\sqrt{d}} + \mathrm{bias}(z[j,k])
$$

That bias term is what makes it a *triangle* attention rather than a plain one: how much
`(i,j)` attends to `(i,k)` depends on the edge `(j,k)` that closes their triangle.

### Rep 3 — `triangle_attention(q, k, v, edge_bias)`
`q, k, v` are `[B, L, L, h, d]` (batch, rows `i`, columns, heads, head-dim). `edge_bias`
is `[B, L, L, h]`, indexed by the **third edge** `(j, k)`. For each row `i`, attend over
the last axis (`k`): `softmax_k( q[i,j]·k[i,k]/√d + edge_bias[j,k] ) · v[i,k]`. Return
`[B, L, L, h, d]`.

In [ ]:
def triangle_attention(q, k, v, edge_bias):
    '''Starting-node triangle attention: attention over k, biased by the (j,k) edge.'''
    # YOUR CODE HERE
    # hint: d = q.size(-1)
    #       logits = einsum('bijhd,bikhd->bijkh', q, k) / sqrt(d)
    #       logits = logits + edge_bias[:, None]        # bias depends on (j,k), broadcast over i
    #       att = softmax(logits, dim=3)                # over k
    #       out = einsum('bijkh,bikhd->bijhd', att, v)
    raise NotImplementedError

# --- checkpoint ---
B_, h, d = 2, 4, 8
q, k, v = (torch.randn(B_, L, L, h, d) for _ in range(3))
zb = torch.zeros(B_, L, L, h)
out = triangle_attention(q, k, v, zb)
assert out.shape == (B_, L, L, h, d), 'wrong shape'
# a huge bias on the third edge (j=5, k=9) forces every (i,5) to attend to (i,9)
bias = torch.zeros(B_, L, L, h); bias[:, 5, 9] = 50.0
outb = triangle_attention(q, k, v, bias)
assert torch.allclose(outb[:, :, 5], v[:, :, 9], atol=1e-3), 'the third-edge bias must steer attention'
print('triangle attention ok — the closing edge (j,k) provably steers the attention ✓')

## Part 4 — the payoff: resolving the indirect couplings of rung 01

Time to settle the score with mutual information. We build a dataset of **many** toy
proteins, each with a random contact graph (so each has its own indirect couplings —
non-contact pairs `(i, j)` that share a common neighbour). For each, we compute the
APC-corrected coevolution matrix exactly as in rung 01 — that is the input — and the
true contact map — that is the target.

Then we train two pair refiners with the same budget:

- **triangle** — a stack of triangle multiplicative updates (your Rep 2),
- **per-pair** — the honest stand-in for a pairwise method: an MLP that processes each
  `z[i, j]` *in isolation*, never comparing it to other pairs.

On held-out proteins we measure contact precision, and — the number that matters — how
many **indirect couplings** each model wrongly promotes into its top predictions. A
pairwise method cannot help but be fooled by them. A triangle-based one can learn not
to be.

In [ ]:
def make_protein():
    ncon = int(rng.integers(10, 16))
    cand = [(i, j) for i in range(L) for j in range(i + 3, L)]; rng.shuffle(cand)
    con = set()
    for (i, j) in cand:
        if len(con) >= ncon: break
        con.add((i, j))
    return con

def coevolution_features(con):
    '''APC-corrected MI matrix [L,L] from an MSA that couples the given contacts.'''
    prof = np.array([rng.dirichlet(np.ones(Q) * 0.8) for _ in range(L)])
    perm = {p: rng.permutation(Q) for p in con}
    m = np.zeros((Nseq, L), int)
    for p in range(L): m[:, p] = rng.choice(Q, size=Nseq, p=prof[p])
    for (i, j) in sorted(con):
        cp = rng.random(Nseq) < 0.8; m[cp, j] = perm[(i, j)][m[cp, i]]
    oh = np.eye(Q)[m]; ps = 0.5; fi = (oh.sum(0) + ps) / (Nseq + Q * ps); M = np.zeros((L, L))
    for i in range(L):
        for j in range(i + 1, L):
            fij = (oh[:, i].T @ oh[:, j] + ps / Q) / (Nseq + ps); fij /= fij.sum()
            M[i, j] = M[j, i] = (fij * np.log(fij / (np.outer(fi[i], fi[j]) + 1e-12) + 1e-12)).sum()
    mm = M.copy(); np.fill_diagonal(mm, 0); col = mm.sum(1, keepdims=True) / (L - 1)
    a = mm[~np.eye(L, dtype=bool)].mean(); c = mm - (col @ col.T) / a; np.fill_diagonal(c, 0)
    return c

def indirect_mask(con):
    '''non-contact pairs with a common neighbour (2-hop) — where coevolution lies.'''
    adj = {i: set() for i in range(L)}
    for (i, j) in con: adj[i].add(j); adj[j].add(i)
    ind = np.zeros((L, L), bool)
    for i in range(L):
        for j in range(i + 3, L):
            if (i, j) not in con and (adj[i] & adj[j]): ind[i, j] = ind[j, i] = True
    return ind

def build_dataset(nprot):
    X, Y, I = [], [], []
    for _ in range(nprot):
        con = make_protein(); X.append(coevolution_features(con))
        tm = np.zeros((L, L), bool)
        for (i, j) in con: tm[i, j] = tm[j, i] = True
        Y.append(tm); I.append(indirect_mask(con))
    return (torch.tensor(np.array(X), dtype=torch.float32),
            torch.tensor(np.array(Y), dtype=torch.float32),
            torch.tensor(np.array(I)))

t0 = time.time()
Xtr, Ytr, Itr = build_dataset(300)
Xte, Yte, Ite = build_dataset(60)
print('built %d train + %d test proteins in %.1fs  (%.1f indirect pairs/protein to be fooled by)'
      % (len(Xtr), len(Xte), time.time() - t0, Ite.sum().item() / 2 / len(Xte)))

In [ ]:
class TriMul(nn.Module):
    '''One triangle multiplicative update block, wrapping your triangle_update core.'''
    def __init__(self, c):
        super().__init__(); self.ln = nn.LayerNorm(c); self.lno = nn.LayerNorm(c)
        self.a, self.b = nn.Linear(c, c), nn.Linear(c, c)
        self.ag, self.bg = nn.Linear(c, c), nn.Linear(c, c)
        self.o, self.og = nn.Linear(c, c), nn.Linear(c, c)
    def forward(self, z):
        zz = self.ln(z)
        a = torch.sigmoid(self.ag(zz)) * self.a(zz)
        b = torch.sigmoid(self.bg(zz)) * self.b(zz)
        out = self.lno(triangle_update(a, b))
        return z + torch.sigmoid(self.og(zz)) * self.o(out)

class Refiner(nn.Module):
    def __init__(self, c=16, nb=3, triangle=True):
        super().__init__(); self.tri = triangle; self.inp = nn.Linear(1, c)
        if triangle:
            self.blocks = nn.ModuleList([TriMul(c) for _ in range(nb)])
        else:   # per-pair baseline: each z[i,j] refined ALONE (no cross-pair mixing)
            self.blocks = nn.ModuleList([nn.Sequential(
                nn.LayerNorm(c), nn.Linear(c, 2 * c), nn.GELU(), nn.Linear(2 * c, c)) for _ in range(nb)])
        self.head = nn.Linear(c, 1)
    def forward(self, x):
        z = self.inp(x[..., None])
        for blk in self.blocks:
            z = blk(z) if self.tri else z + blk(z)
        return self.head(z).squeeze(-1)

UM = torch.triu(torch.ones(L, L), diagonal=3).bool()      # scored pairs (|i-j|>=3, upper tri)

def train_refiner(triangle, steps=250, seed=1):
    torch.manual_seed(seed); net = Refiner(triangle=triangle)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-3)
    pos = Ytr[:, UM]; w = (pos.numel() - pos.sum()) / pos.sum()      # balance sparse contacts
    for st in range(steps):
        idx = torch.randint(0, len(Xtr), (32,))
        logit = net(Xtr[idx])[:, UM]
        loss = F.binary_cross_entropy_with_logits(logit, Ytr[idx][:, UM], pos_weight=w)
        opt.zero_grad(); loss.backward(); opt.step()
    return net

t0 = time.time()
net_tri = train_refiner(triangle=True)
net_pp = train_refiner(triangle=False)
print('trained both refiners in %.0fs' % (time.time() - t0))

### Rep 4 — `evaluate(net)`
For each test protein, take the top-`k` scored pairs (`k` = its number of true
contacts) and return two averages over proteins: **precision** (fraction of the top-`k`
that are real contacts) and **indirect rate** (fraction of the top-`k` that are indirect
couplings — the mistakes we care about). Use the given `UM` pair mask.

In [ ]:
@torch.no_grad()
def evaluate(net):
    '''-> (mean precision@k, mean fraction-of-top-k-that-are-indirect) over test proteins.'''
    # YOUR CODE HERE
    # hint: P = torch.sigmoid(net(Xte)); for each protein b:
    #   s = P[b][UM]; t = Yte[b][UM].bool(); ind = Ite[b][UM].bool(); k = int(t.sum())
    #   top = torch.topk(s, k).indices ; precision += t[top].float().mean()
    #   indirect += ind[top].float().mean()
    raise NotImplementedError

# --- checkpoint ---
prec_tri, ind_tri = evaluate(net_tri)
prec_pp, ind_pp = evaluate(net_pp)
assert prec_tri > prec_pp + 0.05, 'triangle ops should clearly win on precision'
assert ind_tri < ind_pp - 0.05, 'triangle ops should be fooled by FEWER indirect couplings'
print('               precision@L    indirect couplings in top-k')
print('triangle          %.3f              %.3f' % (prec_tri, ind_tri))
print('per-pair (MI-like) %.3f              %.3f' % (prec_pp, ind_pp))
print('\ntriangle wins precision by %+.3f and is fooled %.1fx less often by indirect'
      % (prec_tri - prec_pp, ind_pp / max(ind_tri, 1e-6)))
print('couplings — the exact failure that sank mutual information in rung 01. ✓')

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].bar(['per-pair', 'triangle'], [prec_pp, prec_tri], color=['#cbd5e1', BLUE])
axes[0].set_ylabel('precision@L'); axes[0].set_title('contact precision', fontsize=10)
axes[1].bar(['per-pair', 'triangle'], [ind_pp, ind_tri], color=['#cbd5e1', GREEN])
axes[1].set_ylabel('indirect couplings in top-k'); axes[1].set_title('indirect-coupling mistakes (lower better)', fontsize=10)
for ax in axes: ax.grid(alpha=.15, axis='y')
plt.tight_layout(); plt.show()

## Reflection — what just transferred

- **The pair representation is where geometry lives**, and the **outer-product mean** is
  how the MSA first writes into it — the learned, multi-channel version of rung 01's
  joint frequencies.
- **The triangle multiplicative update is the heart of the trunk.** Rebuilding every
  edge `(i,j)` from the triangles it forms with all `k` is a differentiable triangle
  inequality: an edge must be *consistent* with the edges that close its triangles.
- **Triangle attention** does the same content-dependently, with the closing edge
  `(j,k)` biasing the attention.
- **This is the fix for indirect couplings.** You proved it: a per-pair model — the
  honest stand-in for MI/APC — is fooled by transitive `A–B–C` correlations 3–4× more
  often than the triangle model, which learns that a two-hop path *explains away* the
  phantom edge. The cliffhanger from rung 01 is resolved, and it took reasoning over
  triples to do it.
- These operations are `O(L³)` — the single most expensive part of AlphaFold, and the
  reason its trunk is heavy. The payoff is a pair representation you can trust.

**Next rung:** `AF2·04 — The Evoformer block + distogram head`. We assemble the MSA
attention of rung 02 and the triangle ops of rung 03 into one **Evoformer block**,
stack it into the trunk, and decode an actual **distance histogram** from the pair
representation — a checkable geometric prediction, and the last stop before we turn
pairs into 3D coordinates.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def outer_product_mean(m):
    return torch.einsum('nic,njd->ijcd', m, m) / m.shape[0]

def triangle_update(a, b):
    outgoing = torch.einsum('bikc,bjkc->bijc', a, b)      # i,j share connections to k
    incoming = torch.einsum('bkic,bkjc->bijc', a, b)
    return outgoing + incoming

def triangle_attention(q, k, v, edge_bias):
    d = q.size(-1)
    logits = torch.einsum('bijhd,bikhd->bijkh', q, k) / math.sqrt(d)
    logits = logits + edge_bias[:, None]                  # bias from edge (j,k), broadcast over i
    att = F.softmax(logits, dim=3)                        # over the third vertex k
    return torch.einsum('bijkh,bikhd->bijhd', att, v)

@torch.no_grad()
def evaluate(net):
    P = torch.sigmoid(net(Xte)); precs, inds = [], []
    for b in range(len(Xte)):
        s = P[b][UM]; t = Yte[b][UM].bool(); ind = Ite[b][UM].bool(); k = int(t.sum())
        top = torch.topk(s, k).indices
        precs.append(t[top].float().mean().item())
        inds.append(ind[top].float().mean().item())
    return float(np.mean(precs)), float(np.mean(inds))

print('reference solutions loaded — re-run the checkpoint cells above')